# YOLO11n — V3 865-photo utensil/wrapper supplement
**New V3 trial; V1 frozen specialists and trained V2 are untouched.** This experiment has 697 train / 97 val / 71 test photos. 621 utensil and 140 wrapper boxes overall. Four bottle annotations removed; a missing fork was added to one test image. The annotations are provisional, not exhaustively human-reviewed.

**Change runtime type to T4 GPU** before running. You need `YOLO_V3_865_legacy_supplement.zip` (not the older 704 ZIP).


## 1. Clone the V3 experiment and install dependencies


In [ ]:
import subprocess, sys, torch
from pathlib import Path
assert torch.cuda.is_available(), 'Colab GPU required: Runtime > Change runtime type > T4 GPU'
repo = Path('/content/yolo-cafeteria-research')
if not repo.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/cx-57/yolo-cafeteria-research.git',str(repo)],check=True)
exp = repo / 'experiments/yolo-v3-865'
assert (exp / 'train_yolo_v3.py').is_file()
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(exp/'requirements.txt')],check=True)
print('GPU:',torch.cuda.get_device_name(0))


## 2. Upload the verified V3 dataset ZIP
Checks original 151 MB archive hash, ZIP member structure and decompression integrity.


In [ ]:
import zipfile, hashlib
from google.colab import files
filename = 'YOLO_V3_865_legacy_supplement.zip'
expected_sha = '5138749bbed039886bab1f3ab961ae9430d76a8c36ba7b8e4e57a4f910b4139a'
dataset_yaml = Path('/content/yolo_v3_865_legacy/data.yaml')
if not dataset_yaml.exists():
    uploaded = files.upload()
    assert filename in uploaded, f'Upload {filename}; the older 704 ZIP is incorrect'
    archive = Path('/content') / filename
    if not archive.exists(): archive.write_bytes(uploaded[filename])
    digest = hashlib.sha256()
    with archive.open('rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''): digest.update(chunk)
    assert digest.hexdigest()==expected_sha, 'Source ZIP hash mismatch'
    with zipfile.ZipFile(archive) as z:
        assert z.testzip() is None, 'Corrupt ZIP member'
        assert all(Path(m.filename).parts[:1] == ('yolo_v3_865_legacy',) for m in z.infolist()), 'Unexpected archive member'
        z.extractall('/content')
assert dataset_yaml.is_file(), 'V3 dataset did not extract correctly'
print('Dataset:',dataset_yaml)


## 3. Verify 865 images, 865 labels, object counts and source-group isolation
Training stops if the exact original images, labels or source membership changed.


In [ ]:
script = exp / 'train_yolo_v3.py'
output = Path('/content/YOLO_V3_trial_results')
cmd = [sys.executable,str(script),'--data',str(dataset_yaml),'--output',str(output)]
subprocess.run(cmd+['--verify-only'],check=True)


## 4. Train new YOLO11n for up to 20 epochs and evaluate test
Uses pretrained `yolo11n.pt` and validation-based early stopping. Test is a historically exposed provisional cohort, not a sealed field test.


In [ ]:
assert not output.exists(), f'{output} already exists; do not overwrite a prior run'
subprocess.run(cmd+['--device','0','--batch','8','--epochs','20'],check=True)


## 5. Download the new model and recorded evaluation
Your Colab runtime is temporary. Save the resulting ZIP to your Mac after the run completes.


In [ ]:
import json
report=output/'test_summary.json'
assert report.is_file(), 'Test evaluation did not complete'
print(json.dumps(json.loads(report.read_text()),indent=2))
outzip=Path('/content/YOLO_V3_865_trained_results.zip')
with zipfile.ZipFile(outzip,'w',zipfile.ZIP_DEFLATED) as z:
    for rel in ['test_summary.json','experiment_config.json','resolved_data.yaml','train/weights/best.pt','train/results.csv','train/args.yaml']:
        p=output/rel
        if p.is_file(): z.write(p,arcname=rel)
files.download(str(outzip))
